# Maintenance-need model

Learns **P(a corrective work order was needed)** for every Malaysian tower from
site environment — observed surface water, soil, vegetation, terrain and grid
distance — and reports it against a confusion matrix.

## Read this before any number below

**The labels are synthetic.** No operator or MCMC maintenance history was
obtainable for this prototype, so `data/malaysia/maintenance_records.csv` is
simulated by `data/prepare_maintenance_records.py`. Everything here therefore
validates the **evaluation pipeline**, not real-world accuracy, and every figure
will change when real work orders arrive.

**The generator was retuned for accuracy, and that decides how to read the
score.** The planted traps — an interaction, a non-monotone response, a
threshold cliff, a per-state contractor effect and a leakage-bait column — have
been removed, the hazard forms are now smooth and monotone in the observed
columns, the latent site attributes are attenuated to `LATENT_STRENGTH = 0.15`
of their natural spread, and reporting noise is off. The model scores far
better than it did before (ROC ~0.57 -> ~0.95) and **none of that improvement
came from the model**: the label was made more learnable. This number measures
how learnable this generator made itself, not how well maintenance need can be
predicted from satellite data.

Two guards survive the retune and still do real work:

1. **The generator never imports the physical index.** Hazard is driven by
   observed measurements through functional forms that are ours — ramps and
   exponentials — never the index's logistic memberships combined by noisy-OR.
   Pinned by an AST test.
2. **An oracle ceiling is reported.** The generator knows each site's true
   latent intensity. Scoring it bounds what *any* model could reach here. A
   model that lands at the oracle has not succeeded; the generator has leaked.

Demo cases are still registered in `demo_cases.json` before this notebook runs,
so "the model fell for this" stays a finding rather than a story assembled after
seeing predictions.

This is **not** a failure predictor. The target is that a work order was raised.
It is not an outage, a fault, or a probability of one.

Generator v2.3 deliberately adds measured EVI regrowth to the synthetic vegetation cause.
This grades our own generator, not real maintenance benefit. The observation ledger
collects evidence only; confirmation and retraining are separate, manual steps.


In [ ]:
import json
from datetime import datetime, date, timezone
import sys
from pathlib import Path

BACKEND = Path.cwd().parent / "src" / "backend"
sys.path.insert(0, str(BACKEND))
sys.path.insert(0, str(BACKEND / "data"))

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score as roc_auc
from sklearn.model_selection import GroupKFold, KFold

# Every shared constant is IMPORTED, never redefined here. A notebook that
# declares its own FEATURES trains a model the API then feeds different columns
# to, in a different order, and LightGBM accepts a positional array without
# complaint. model/test_maintenance_need.py parses this notebook and fails the
# build on any redefinition.
from model.maintenance_need import (
    CATEGORICAL_FEATURES, EARLY_STOPPING_ROUNDS, FACTOR_GROUPS, FACTORS,
    FEATURES, MODEL_PATH, NUM_BOOST_ROUND, PARAMS, REPORT_PATH, TOP_K,
    build_matrix, dominant_factor, factor_shares,
)
from model.ensemble import CHANGE_BLEND_WEIGHT, CONDITION_BLEND_WEIGHT, blend_priority
from model.change import change_scores
import prepare_maintenance_records as generator
from scheduler.config_loader import load_policy
from model.novelty import IF_PARAMS, NOVELTY_FEATURES, TELEMETRY_COLUMNS, condition_scores
from model.profiler import BehavioralProfiler
from model.flood_eval import confusion, ranking_metrics, top_k_confusion
from model.risk_index import AGE_WEIGHT, AHP_FACTORS, ahp_weights, memberships, noisy_or
from prepare_pilot_dataset import write_json

DATA = BACKEND.parent.parent / "data" / "malaysia"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})
print("features:", len(FEATURES), "| factors:", FACTORS, "| top-k:", TOP_K)

## 1. Load

The feature table, the Earth Engine land features, and the synthetic labels.
The base-rate assertion is deliberate: an out-of-range base rate makes every
figure below it meaningless, so the notebook stops here rather than producing
plots nobody should trust.

In [ ]:
towers = pd.read_csv(DATA / "tower_feature_table.csv")
land = pd.read_csv(DATA / "land_features.csv")
labels = pd.read_csv(DATA / "maintenance_labels.csv")
manifest = json.loads((DATA / "maintenance_manifest.json").read_text())
demo = json.loads((DATA / "demo_cases.json").read_text())

df = (towers.merge(land, on="tower_id")
            .merge(labels.drop(columns=["state"]), on="tower_id"))
df["radio"] = df["radio"].fillna("UNKNOWN")
df["state"] = df["state"].fillna("unassigned").replace("", "unassigned")

y = df["needed_corrective_maintenance"].to_numpy()
groups = df["state"].to_numpy()
X = build_matrix(df)

base_rate = float(y.mean())
assert manifest["synthetic"] is True, "labels must declare themselves synthetic"
assert 0.15 <= base_rate <= 0.30, f"base rate {base_rate:.3f} out of usable range"

print(f"{len(df)} towers | {int(y.sum())} positives | base rate {base_rate:.3f}")
print(f"{df.state.nunique()} states | window {manifest['window'][0]} to {manifest['window'][1]}")
print(f"rejected malformed tickets: {manifest['rejected_tickets']}")

## 2. Look at the data before modelling it

The third panel is the one worth staring at. `gsw_occurrence_pct` is the mapped
water history, and it is 0 for ~98% of these towers. Since the v2.0 retune
`hand_m` **also** drives flood hazard directly (a shape-1.5 decay, not the
index's logistic membership), which is a large part of why the model now scores
where it does: before the retune the only route from HAND to flood risk was
whatever correlation this panel shows, and the model had to learn it.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))

rate = df.groupby("state")["needed_corrective_maintenance"].agg(["mean", "size"])
rate = rate[rate["size"] >= 20].sort_values("mean")
ax[0].barh(rate.index, rate["mean"], color="#4c72b0")
ax[0].axvline(base_rate, color="#c44e52", ls="--", lw=1, label=f"national {base_rate:.2f}")
ax[0].set_title("corrective rate by state (n>=20)")
ax[0].legend(fontsize=7)

ax[1].hist(df.loc[y == 0, "soil_moisture_p90"].dropna(), bins=30, alpha=0.6, label="no order", density=True)
ax[1].hist(df.loc[y == 1, "soil_moisture_p90"].dropna(), bins=30, alpha=0.6, label="order raised", density=True)
ax[1].set_title("soil moisture p90")
ax[1].legend(fontsize=7)

wet = df["gsw_occurrence_pct"] > 0
ax[2].scatter(df.loc[~wet, "hand_m"], df.loc[~wet, "gsw_occurrence_pct"], s=6, alpha=0.3, label="never observed wet")
ax[2].scatter(df.loc[wet, "hand_m"], df.loc[wet, "gsw_occurrence_pct"], s=10, alpha=0.7, color="#c44e52", label="observed wet")
ax[2].set_xscale("symlog"); ax[2].set_xlabel("hand_m"); ax[2].set_ylabel("GSW occurrence %")
ax[2].set_title("what the model must learn")
ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"towers ever observed under water: {int(wet.sum())} of {len(df)} ({wet.mean():.1%})")

## 3. Cross-validation

**GroupKFold on ADM1 state, never a random split.** Towers a few kilometres
apart share a catchment, a grid feeder and a maintenance crew, so a random split
puts near-duplicates in both train and test. The per-state contractor multiplier
that used to make this leak severe was removed in the v2.0 retune, so the two
splits should now agree — checked, not assumed, in section 6.

In [ ]:
def out_of_fold(X, y, groups, splitter, params=PARAMS):
    # Pooled out-of-fold probabilities. Safe to pool: every row is scored by a
    # fit that never saw its group.
    out = np.full(len(y), np.nan)
    boosters = []
    for train_idx, test_idx in splitter.split(X, y, groups):
        # An inner split for early stopping, so no fold picks its own round
        # count against the rows it is scored on.
        cut = int(len(train_idx) * 0.85)
        fit_idx, val_idx = train_idx[:cut], train_idx[cut:]
        booster = lgb.train(
            params,
            lgb.Dataset(X.iloc[fit_idx], y[fit_idx], categorical_feature=CATEGORICAL_FEATURES),
            num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[lgb.Dataset(X.iloc[val_idx], y[val_idx], categorical_feature=CATEGORICAL_FEATURES)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        out[test_idx] = booster.predict(X.iloc[test_idx])
        boosters.append(booster)
    return out, boosters


n_splits = min(5, df.state.nunique())
grouped = GroupKFold(n_splits=n_splits)
oof, fold_boosters = out_of_fold(X, y, groups, grouped)
print(f"out-of-fold over {n_splits} state-grouped folds; {np.isnan(oof).sum()} rows unscored")

## 4. Four scorers, one table

| scorer | what it is |
|---|---|
| `lightgbm` | the model, scored out of fold |
| `ahp_index` | the noisy-OR physical index — the baseline being replaced |
| `oracle` | the generator's own latent intensity — the ceiling |
| `hand_only` | a single feature, as a sanity floor |

`oracle` is what makes the other numbers readable. It is not a competitor.

In [ ]:
p_index = memberships(df).drop(columns=["lightning"], errors="ignore")
w0, _ = ahp_weights()
index_weights = dict(zip(AHP_FACTORS, w0 / w0.max()))
index_weights["age"] = AGE_WEIGHT
index_risk = noisy_or(p_index, {c: index_weights[c] for c in p_index.columns})
assert np.isfinite(index_risk).all(), "index baseline must not carry NaN"

scorers = {
    "lightgbm": oof,
    "ahp_index": index_risk,
    "oracle": df["lambda_true"].to_numpy(),
    "hand_only": -df["hand_m"].to_numpy(),   # lower HAND = nearer drainage
}

rows = []
for name, score in scorers.items():
    rows.append({
        "scorer": name,
        "ranking": ranking_metrics(y, score),
        "at_top_10_percent": top_k_confusion(y, score, TOP_K),
    })

header = f"{'scorer':12s} {'ROC':>6s} {'PR':>6s} {'TP':>4s} {'FP':>5s} {'FN':>4s} {'TN':>5s} {'prec':>6s} {'rec':>6s} {'lift':>5s}"
print(header); print("-" * len(header))
for r in rows:
    t, k = r["at_top_10_percent"], r["ranking"]
    print(f"{r['scorer']:12s} {k['roc_auc']:>6} {k['pr_auc']:>6} {t['true_positive']:>4d} "
          f"{t['false_positive']:>5d} {t['false_negative']:>4d} {t['true_negative']:>5d} "
          f"{t['precision']:>6} {t['recall']:>6} {t['lift_over_random']:>5}")
print(f"\n(TP/FP/FN/TN at the top-{int(TOP_K*100)}% operating point — the slice crew capacity actually dispatches)")

### Reading the table, including the leak check

`lightgbm` must beat `ahp_index` and sit **meaningfully below** `oracle`. If it
were within ~0.03 PR-AUC of the oracle, the generator would have leaked
recoverable structure and the whole evaluation would be void — that check runs
below and is not decorative.

In [ ]:
pr = {r["scorer"]: r["ranking"]["pr_auc"] for r in rows}
gap_to_oracle = pr["oracle"] - pr["lightgbm"]
beats_index = pr["lightgbm"] - pr["ahp_index"]

print(f"lightgbm PR-AUC   {pr['lightgbm']:.4f}")
print(f"ahp_index PR-AUC  {pr['ahp_index']:.4f}   (delta {beats_index:+.4f})")
print(f"oracle PR-AUC     {pr['oracle']:.4f}   (gap {gap_to_oracle:+.4f})")
print()
if gap_to_oracle < 0.03:
    print("LEAK SUSPECTED: the model is at the oracle. The generator has handed")
    print("over recoverable structure; these figures do not mean anything.")
elif beats_index <= 0:
    print("FINDING: the supervised model does NOT beat the physical index here.")
    print("Reported as-is rather than tuned away — see the notes at the end.")
else:
    print("Model beats the index and stays clear of the ceiling, as required.")

## 5. Attribution

Refit on all rows, then exact TreeSHAP grouped into the five factors the
scheduler speaks in. Shares are normalised **absolute** contributions: raw SHAP
is signed, and a factor that lowers a site's risk would otherwise render as a
negative bar in a panel that only draws positives.

In [ ]:
full = lgb.train(
    PARAMS,
    lgb.Dataset(X, y, categorical_feature=CATEGORICAL_FEATURES),
    num_boost_round=int(np.median([b.best_iteration or NUM_BOOST_ROUND for b in fold_boosters])),
)
shares = factor_shares(full, X)
dominant = dominant_factor(shares)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
shares.mean().sort_values().plot.barh(ax=ax[0], color="#55a868")
ax[0].set_title("mean attribution share")
dominant.value_counts().plot.bar(ax=ax[1], color="#4c72b0", rot=0)
ax[1].set_title("dominant factor, tower count")
plt.tight_layout(); plt.show()

worked = int(np.argmax(oof))
print(f"worked example — {df.tower_id.iloc[worked]} ({df.state.iloc[worked]})")
print(f"  risk {oof[worked]:.3f} | dominant {dominant.iloc[worked]} | actual orders {df.n_corrective_36mo.iloc[worked]}")
for factor, share in shares.iloc[worked].sort_values(ascending=False).items():
    print(f"    {factor:11s} {share:.3f}")

## 6. Split check

The per-state contractor effect that used to make this a trap is gone, so a
random split and a grouped split should now broadly **agree**. Reported anyway,
because the day real records arrive they will carry regional structure again and
this is the check that catches it.


### Random KFold vs GroupKFold by state

A random split puts towers from the same state in both train and test. When the
generator plants no state-level effect there is nothing for it to collect credit
for, and the two bars should sit close together. A large gap means state
structure has crept back in — from the data, not from the model — and the
grouped split is the honest one.


In [ ]:
random_oof, _ = out_of_fold(X, y, groups, KFold(n_splits=n_splits, shuffle=True, random_state=0))
random_pr = ranking_metrics(y, random_oof)["pr_auc"]
grouped_pr = pr["lightgbm"]
inflation = random_pr - grouped_pr

fig, ax = plt.subplots(figsize=(4.2, 3))
ax.bar(["random\nKFold", "GroupKFold\nby state"], [random_pr, grouped_pr],
       color=["#c44e52", "#4c72b0"])
ax.set_ylabel("PR-AUC"); ax.set_title(f"split difference {inflation:+.3f}")
plt.tight_layout(); plt.show()

print("GroupKFold on state is reported throughout regardless of the gap.")


## 7. Planted demo cases

Registered before training, with `expected_role` stated up front — so this is a
test, not a story. Three kinds of expectation, checked three different ways,
because they are not the same claim:

* **confusion** — `protected_site` should be a false positive, `neglected_site`
  a false negative. Compares straight against the observed confusion cell.
* **pair** — a `twin_pair` is matched so the model *cannot* separate its two
  members, so neither can be judged alone. The claim is about the pair: near
  identical scores, opposite outcomes. A pair the model cannot separate can only
  land {TN, FN} or {FP, TP}; expecting one TN and one TP is not a stricter test,
  it is an impossible one.
* **narrative** — `count_is_not_cost`, `denominator_matters` and the rest name a
  lesson, not a cell of the confusion matrix. Reported, never scored: comparing
  them to `observed_role` is false by construction.


In [ ]:
cut = float(np.quantile(oof, 1 - TOP_K))
lookup = {t: i for i, t in enumerate(df.tower_id)}

outcomes = []
for case in demo["cases"]:
    i = lookup.get(case["tower_id"])
    if i is None:
        continue
    dispatched, actual = bool(oof[i] >= cut), bool(y[i])
    outcomes.append({
        "tower_id": case["tower_id"], "case_type": case["case_type"],
        "expected_role": case["expected_role"],
        "expectation_kind": case.get("expectation_kind", "narrative"),
        "partner_id": case.get("partner_id"),
        "risk": round(float(oof[i]), 4),
        "in_top_decile": dispatched, "actually_needed": actual,
        "observed_role": ("true_positive" if dispatched and actual else
                          "false_positive" if dispatched else
                          "false_negative" if actual else "true_negative"),
    })

table = pd.DataFrame(outcomes)

# --- confusion cases: direct comparison ------------------------------------
conf = table[table.expectation_kind == "confusion"]
held = int((conf.observed_role == conf.expected_role).sum())
print(f"confusion cases: {held}/{len(conf)} landed in their expected cell")
for _, r in conf.iterrows():
    mark = "as expected" if r.observed_role == r.expected_role else "DIFFERED"
    print(f"   {r.case_type:16s} {r.tower_id:18s} risk {r.risk:.3f}  {r.observed_role:15s} <- {mark}")

# --- pair cases: judged as pairs, never singly ------------------------------
# SEPARATION_TOL is on the model's probability scale. The pair is planted to be
# indistinguishable in feature space; if the model still separates them by more
# than this it has found something the twin match missed, and that is a finding
# about the match, not about the model.
SEPARATION_TOL = 0.10
pairs, seen = [], set()
by_id = {r["tower_id"]: r for r in outcomes}
for r in outcomes:
    if r["expectation_kind"] != "pair" or r["tower_id"] in seen:
        continue
    other = by_id.get(r["partner_id"])
    if other is None:
        continue
    seen.update({r["tower_id"], other["tower_id"]})
    gap = abs(r["risk"] - other["risk"])
    pairs.append({
        "a": r["tower_id"], "b": other["tower_id"],
        "risk_a": r["risk"], "risk_b": other["risk"], "score_gap": round(gap, 4),
        "outcomes_differ": r["actually_needed"] != other["actually_needed"],
        "scores_close": gap <= SEPARATION_TOL,
        "held": (gap <= SEPARATION_TOL) and (r["actually_needed"] != other["actually_needed"]),
    })

print(f"\ntwin pairs: {sum(p['held'] for p in pairs)}/{len(pairs)} show the ceiling "
      f"(scores within {SEPARATION_TOL}, opposite outcomes)")
for p in pairs:
    why = "ceiling shown" if p["held"] else (
        "model separated them" if not p["scores_close"] else "same outcome, no contrast")
    print(f"   {p['a']:18s} {p['risk_a']:.3f}  vs  {p['b']:18s} {p['risk_b']:.3f}"
          f"   gap {p['score_gap']:.3f}  <- {why}")

# --- narrative cases: reported, never scored --------------------------------
print("\nnarrative cases (reported, not scored — the role names a lesson, "
      "not a confusion cell):")
for _, r in table[table.expectation_kind == "narrative"].iterrows():
    print(f"   {r.case_type:16s} {r.expected_role:20s} risk {r.risk:.3f}  {r.observed_role}")


## 9. The other two layers

Two more signals, and the section exists to decide whether either belongs in the
served score. Neither is allowed to move `risk` without earning it here.

* **Layer 2 — Isolation Forest** over the same site environment, minus the
  categorical. Unsupervised, so it needs no label: it asks how far a site sits
  from the cloud of sites the model was fitted on. That is a statement about the
  *training distribution*, not about the tower, and it is the one question a
  gradient-boosted model structurally cannot answer — LightGBM interpolates
  inside the region its rows covered and says nothing about anything outside it.
* **Layer 3 — BehavioralProfiler**, deterministic rules over context that has no
  feature column: whether a crew can reach the site, whether it was scored on a
  complete row, whether a live forecast couples to its dominant factor.

**The forest is refit inside every fold, on training rows only.** Fitting once
over all towers leaks the held-out fold's *geometry* into the anomaly boundary.
No label is involved, which is exactly what makes it easy to miss, and it
inflates the standalone number below.

**Layer 3 may not read maintenance history, and that is not caution.** The
served label is the site's own corrective-ticket count thresholded at one — both
select the same 246 rows in this dataset — so a rule reading that history
predicts the target by construction, reports near-perfect accuracy, and measures
arithmetic. It is the failure that withdrew `maintenance_classifier`. A
repeat-visit rule is the most natural rule to want here and it is the one that
cannot exist in this framing; `model/test_ensemble.py` parses the module's AST
and fails the build on those column names.

In [ ]:
def out_of_fold_novelty(X, groups, splitter):
    """Per-fold isolation forest, fit on TRAIN rows only.

    A single fit over all towers would put the held-out fold's geometry inside
    the boundary that then judges it. Nothing about the label is involved, so
    nothing looks wrong; the standalone number just comes back too good.
    """
    out = np.full(len(X), np.nan)
    Z = X[NOVELTY_FEATURES]
    for train_idx, test_idx in splitter.split(X, groups=groups):
        complete = ~Z.iloc[train_idx].isna().any(axis=1).to_numpy()
        forest = IsolationForest(**IF_PARAMS).fit(Z.iloc[train_idx][complete])
        scorable = ~Z.iloc[test_idx].isna().any(axis=1).to_numpy()
        # score_samples is negated by sklearn's convention: lower = stranger.
        out[np.asarray(test_idx)[scorable]] = -forest.score_samples(
            Z.iloc[test_idx][scorable]
        )
    return out


raw_novelty = out_of_fold_novelty(X, groups, grouped)
scorable = ~np.isnan(raw_novelty)
novelty = pd.Series(raw_novelty).rank(pct=True).to_numpy()   # a rank, not a probability

novelty_ranking = ranking_metrics(y[scorable], novelty[scorable])
model_on_subset = ranking_metrics(y[scorable], oof[scorable])
agreement = float(pd.Series(oof[scorable]).corr(pd.Series(novelty[scorable]), method="spearman"))

print(f"{int(scorable.sum())} of {len(y)} towers scorable "
      f"({int((~scorable).sum())} carry a missing feature — IsolationForest raises on NaN,\n"
      f"and imputing a median there would invent an observation)\n")
header = f"{'scorer':22s} {'ROC':>7s} {'PR':>7s} {'top-10% prec':>13s} {'lift':>6s}"
print(header); print("-" * len(header))
for name, s_ in [("lightgbm", oof), ("isolation_forest", novelty)]:
    t = top_k_confusion(y[scorable], s_[scorable], TOP_K)
    k = ranking_metrics(y[scorable], s_[scorable])
    print(f"{name:22s} {k['roc_auc']:>7} {k['pr_auc']:>7} {t['precision']:>13} {t['lift_over_random']:>6}")
print(f"\nspearman(lightgbm, novelty) = {agreement:+.4f}")
print("Novelty is not noise — but it agrees with the model substantially, which is"
      "\nthe first sign that adding it to the score buys correlation, not information.")

### Which layer may move a decision, and in what SHAPE

Two questions, and the second one cost more to learn than the first.

**Which signal.** `novelty` (isolation forest over site environment) may not: it
scores |deviation from typical| while maintenance need is monotone, so a site
with unusually *few* alarms is as anomalous as one with unusually many. Given a
telemetry block that ranks the label at ROC 0.93 on its own, a forest returns
**~0.52** inside the escalation band. The same column read one-sided returns
**~0.83**. `condition_auc > model_auc` inside the band is the necessary
condition for a second opinion to be worth acting on, and it is checked below.

**What shape.** A GATE — lift a `watch` tower when its condition clears a
threshold — shipped first and is wrong. It touches the second opinion at one
cut and reorders nothing, so it discards what condition says about every other
tower. Measured on held-out seeds under a realistic telemetry-noise model, at a
matched dispatch budget:

| policy | F1 | TP | vs LightGBM | seeds won |
|---|---|---|---|---|
| lightgbm alone | 0.6623 | 138.4 | +0.0000 | |
| gate, p85, cap 5%, watch only | 0.6497 | 135.8 | **−0.0125** | 1/5 |
| gate, p70, cap 20%, watch+ok | 0.6738 | 140.8 | +0.0115 | 5/5 |
| **rank blend, w = 0.25** | **0.6814** | **142.4** | **+0.0191** | **5/5** |

Relaxing the cap changes nothing — it was never what bound the gate. The shape
was. `risk` still stays the model's own probability; the blend produces a
separate `priority` that the bands are cut on.

Anything that dispatches more towers is scored against a **cut that dispatches
the same number**, never against the untouched top-10%. That correction is what
exposed the gate.

**Satellite change is a third candidate rank.** Only positive EVI change counts;
clearing is not encroachment. Its in-band AUC uses the same measured rows as the
model comparison. Missing evidence is null, not a measured zero. Sweep weights
0/0.1/0.15/0.2/0.3 on synthetic seeds 0–4 and verify on 5–9, at matched budget.
The necessary condition must pass before any nonzero change weight can ship.
The ledger records actual windows and comparisons; its outcome gate stays
unlabeled until confirmation. Demo confirmations are marked simulated.


In [ ]:
# --- 1. blending into `risk` — still rejected --------------------------------
rank = lambda v: pd.Series(v).rank(pct=True).to_numpy()
blend_sweep = []
for w in (0.0, 0.1, 0.2, 0.3, 0.5):
    blended = (1 - w) * rank(oof[scorable]) + w * rank(novelty[scorable])
    k = ranking_metrics(y[scorable], blended)
    blend_sweep.append({"w_novelty": w, "roc_auc": k["roc_auc"], "pr_auc": k["pr_auc"]})
print("blending novelty into risk:  " + "  ".join(
    f"w={r['w_novelty']:.1f} PR {r['pr_auc']}" for r in blend_sweep))
print("worse at every weight — `risk` stays the model's own probability.\n")

# --- 2. the necessary condition, per detector --------------------------------
# Label-free ranks describe the same population used to cut dispatch bands,
# matching serving. Only the fitted model needs state-held-out predictions.
telemetry = pd.read_csv(DATA / "site_telemetry.csv")
condition = condition_scores(df.tower_id.to_numpy(), telemetry)
change = change_scores(df.tower_id.to_numpy())

raw_telem_auc = float(roc_auc(y, np.log1p(
    telemetry.set_index("tower_id").reindex(df.tower_id)[TELEMETRY_COLUMNS]).mean(axis=1)))

print(f"raw telemetry ROC against the label, all towers: {raw_telem_auc:.4f}\n")
print(f"{'detector':>22s} {'in-band AUC':>12s} {'model AUC':>10s} {'delta':>8s} {'may gate?':>10s}")
print("-" * 66)
detectors = {}
def detector_comparison(target, predictions, evidence):
    hi, lo = np.quantile(predictions, [0.90, 0.70])
    usable = (predictions < hi) & (predictions >= lo) & np.isfinite(evidence)
    if len(set(target[usable])) < 2:
        return {"in_band_auc": None, "model_auc": None, "delta": None, "may_gate": False}
    auc = float(roc_auc(target[usable], evidence[usable]))
    model_auc = float(roc_auc(target[usable], predictions[usable]))
    return {"in_band_auc": round(auc, 4), "model_auc": round(model_auc, 4),
            "delta": round(auc - model_auc, 4), "may_gate": auc > model_auc}

for name, score in (("isolation_forest", novelty), ("one_sided_condition", condition),
                    ("one_sided_change", change)):
    detectors[name] = detector_comparison(y, oof, score)
    print(name, detectors[name])
print("\nA forest scores |deviation| and maintenance need is monotone: a quiet site")
print("is as anomalous as a loud one, so most of the signal cancels.\n")

# --- 3. the served policy against a matched-budget cut -----------------------
# The adapter's OWN blend, so the notebook and the API cannot disagree about the
# ordering that decides dispatch.
priority = blend_priority(oof, condition, change)
budget = int(round(0.10 * len(y)))
gate = np.zeros(len(y), bool); gate[np.argsort(-priority)[:budget]] = True
lower = np.zeros(len(y), bool); lower[np.argsort(-oof)[:budget]] = True
escalated_mask = gate & ~lower

def confusion_at(selected):
    tp = int((selected & (y == 1)).sum()); fp = int((selected & (y == 0)).sum())
    fn = int((~selected & (y == 1)).sum())
    return {"n": int(selected.sum()), "tp": tp, "fp": fp,
            "precision": round(tp / max(1, tp + fp), 4),
            "recall": round(tp / max(1, tp + fn), 4),
            "f1": round(2 * tp / max(1, 2 * tp + fp + fn), 4)}

lower_cut, ensemble = confusion_at(lower), confusion_at(gate)
print(f"both policies dispatch {budget} towers (blend weight {CONDITION_BLEND_WEIGHT})\n")
print(f"{'policy':>26s} {'TP':>4s} {'FP':>4s} {'precision':>10s} {'recall':>8s} {'F1':>8s}")
print("-" * 62)
for label, r in (("LightGBM alone", lower_cut), ("3-layer ensemble (blend)", ensemble)):
    print(f"{label:>26s} {r['tp']:>4d} {r['fp']:>4d} {r['precision']:>10.4f} "
          f"{r['recall']:>8.4f} {r['f1']:>8.4f}")
delta_f1 = round(ensemble["f1"] - lower_cut["f1"], 4)
print(f"\nensemble minus matched-budget LightGBM:  F1 {delta_f1:+.4f}   "
      f"TP {ensemble['tp'] - lower_cut['tp']:+d}")
swapped = int(escalated_mask.sum())
print(f"the blend swapped {swapped} towers into the dispatch list; "
      f"{int(y[escalated_mask].sum())} of them needed work "
      f"({y[escalated_mask].mean():.3f} against {y[~lower].mean():.3f} below the model's own cut)")

# --- 4. is novelty an out-of-distribution warning instead? -------------------
# If it were, model error would rise with it and abstaining on the strangest
# towers would improve what remains. Neither happens: calibration and ROC stay
# flat while the BASE RATE climbs, so novelty tracks risk, not model error.
quintile = pd.qcut(pd.Series(novelty[scorable]), 5, labels=False).to_numpy()
yy, pp = y[scorable], oof[scorable]
print(f"\n{'novelty quintile':>17s} {'n':>5s} {'base':>6s} {'|calib err|':>12s} {'ROC':>7s}")
print("-" * 50)
ood = []
for q_ in range(5):
    m_ = quintile == q_
    row = {"quintile": q_ + 1, "n": int(m_.sum()), "base_rate": round(float(yy[m_].mean()), 4),
           "calibration_error": round(float(abs(pp[m_].mean() - yy[m_].mean())), 4),
           "roc_auc": round(float(roc_auc(yy[m_], pp[m_])), 4) if len(set(yy[m_])) > 1 else None}
    ood.append(row)
    print(f"{'Q' + str(q_ + 1):>17s} {row['n']:>5d} {row['base_rate']:>6.3f} "
          f"{row['calibration_error']:>12.4f} {row['roc_auc']:>7}")

# --- 5. layer 3 coverage -----------------------------------------------------
from adapter.ml_source import TERRITORY_ALIASES

profiler = BehavioralProfiler()
pseudo = [{"tower_id": t, "lon": float(lo_), "lat": float(la), "radio": r,
           "territory": TERRITORY_ALIASES.get(st, st), "dominant_factor": dominant.iloc[i],
           "attribution": shares.iloc[i].to_dict(), "weather": None,
           "priority": float(priority[i]),
           "decision": ("maintain" if priority[i] >= np.quantile(priority, .90) else
                        "watch" if priority[i] >= np.quantile(priority, .70) else "ok"),
           "condition": None if np.isnan(condition[i]) else float(condition[i]),
           "change": None if np.isnan(change[i]) else float(change[i])}
          for i, (t, lo_, la, r, st) in enumerate(
              zip(df.tower_id, df.lon, df.lat, df.radio, df.state))]
tower_flags = [profiler.flags(rec, X.iloc[i]) for i, rec in enumerate(pseudo)]
print(f"\n{'rule':22s} {'towers':>7s} {'evaluable':>10s} {'lift':>7s}")
print("-" * 49)
flag_coverage = {}
for rule_id, rule in profiler.rules.items():
    fires = np.array([any(f["id"] == rule_id for f in fl) for fl in tower_flags])
    entry = {"towers": int(fires.sum()), "evaluable": bool(rule.get("evaluable", False)), "lift": None}
    if entry["evaluable"] and fires.any():
        entry["lift"] = round(float(y[fires].mean()) / base_rate, 3)
    flag_coverage[rule_id] = entry
    lift = f"{entry['lift']:.2f}x" if entry["lift"] is not None else "-"
    print(f"{rule_id:22s} {entry['towers']:>7d} {str(entry['evaluable']):>10s} {lift:>7s}")

verdict = "ensemble_beats_matched_budget" if delta_f1 > 0 else "descriptive_only"
print(f"\nverdict: {verdict}")

# --- 6. change weight: choose on seeds 0-4, inspect untouched seeds 5-9 -------
# Regenerate synthetic outcomes and telemetry per seed without writing files.
# Every model evaluation holds out ADM1 states; all weights dispatch the same
# number of towers, and delta is against the EXISTING two-rank policy.
change_blend_sweep = []
if np.isfinite(change).any():
    seed_frame = generator.load_inputs(generator.TOWER_TABLE, generator.LAND_FEATURES,
                                       generator.LAND_CHANGE)
    seed_frame = seed_frame.set_index("tower_id").loc[df.tower_id].reset_index()
    policy = load_policy()
    monsoon = policy["monsoon"]["months"]
    end = date.fromisoformat(policy["demo_clock"]["today"]).replace(day=1)
    assert generator.DROP_RATE == generator.SPURIOUS_RATE == 0, "counts-only sweep assumes no reporting noise"
    for seed in range(10):
        rng = np.random.default_rng(seed)
        latents = generator.draw_latents(rng, seed_frame)
        baseline = generator.cause_intensities(seed_frame, latents)
        generator.plant_demo_cases(rng, seed_frame, latents, baseline)
        intensities = generator.cause_intensities(seed_frame, latents)
        scale = generator.calibrate(seed_frame, latents, intensities, monsoon, end, seed)
        _, latent_rate, counts = generator.simulate(
            np.random.default_rng(seed + 1), seed_frame, latents, intensities,
            scale, monsoon, end, record=False)
        seed_y = (counts > 0).astype(int)
        assert abs(seed_y.mean() - generator.TARGET_BASE_RATE) <= generator.BASE_RATE_TOLERANCE
        seed_oof, _ = out_of_fold(X, seed_y, groups, grouped)
        seed_telemetry = generator.build_telemetry(
            np.random.default_rng(seed + 77_000), seed_frame.tower_id.to_numpy(), latent_rate)
        seed_condition = condition_scores(seed_frame.tower_id.to_numpy(), seed_telemetry)
        necessary = detector_comparison(seed_y, seed_oof, change)
        for weight in (0.0, 0.1, 0.15, 0.2, 0.3):
            selected = np.argsort(-blend_priority(seed_oof, seed_condition, change,
                                                change_weight=weight), kind="stable")[:budget]
            tp = int(seed_y[selected].sum())
            f1 = 2 * tp / (budget + int(seed_y.sum()))
            if weight == 0:
                baseline_f1 = f1
            change_blend_sweep.append({
                "seed": seed, "phase": "sweep" if seed < 5 else "verification",
                "w_change": weight, "budget": budget, "tp": tp,
                "f1": round(f1, 4), "delta_f1": round(f1 - baseline_f1, 4),
                "in_band_auc": necessary["in_band_auc"], "model_auc": necessary["model_auc"],
            })
        print("change sweep seed", seed, necessary, flush=True)
    display(pd.DataFrame(change_blend_sweep).groupby(["phase", "w_change"])[["f1", "delta_f1"]].mean())

change_necessary = detectors["one_sided_change"]
assert CHANGE_BLEND_WEIGHT == 0 or change_necessary["may_gate"], "Set CHANGE_BLEND_WEIGHT = 0.0: necessary condition failed"
change_verdict = (
    "Satellite change is unmeasured; descriptive only, weight 0.0."
    if change_necessary["in_band_auc"] is None else
    f"Satellite change in-band AUC {change_necessary['in_band_auc']:.4f} vs model "
    f"{change_necessary['model_auc']:.4f}. " +
    ("Necessary condition failed; descriptive only, weight 0.0." if not change_necessary["may_gate"] else
     f"Necessary condition passed; served change weight {CHANGE_BLEND_WEIGHT}. See the matched-budget seed sweep.")
)
print(change_verdict)


## 10. Persist

The report is the durable artifact. The Method page and the demo read
`maintenance_need_report.json`, never this notebook — a figure that appears here
but not in the report cannot be cited in the UI.

In [ ]:
from prepare_maintenance_records import LATENT_STRENGTH as gen_latent_strength
from prepare_maintenance_records import OBSERVABLE_LOG_SPREAD as gen_log_spread

report = {
    "synthetic_labels": True,
    "caveat": manifest["caveat"],
    "not_a_failure_label": manifest["not_a_failure_label"],
    "generator_version": manifest["generator_version"],
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "encroachment_note": manifest.get("encroachment_note"),
    "dataset": {
        "towers": int(len(df)), "positives": int(y.sum()),
        "base_rate": round(base_rate, 4), "states": int(df.state.nunique()),
        "window": manifest["window"], "rejected_tickets": manifest["rejected_tickets"],
    },
    "cv": {"scheme": "GroupKFold on ADM1 state", "folds": n_splits,
           "operating_point": f"top-{int(TOP_K*100)}%"},
    "scorers": rows,
    "leak_check": {
        "gap_to_oracle": round(float(gap_to_oracle), 4),
        "beats_index_by": round(float(beats_index), 4),
        "verdict": "clear" if gap_to_oracle >= 0.03 else "leak suspected",
    },
    "split_check": {
        "random_kfold_pr_auc": round(float(random_pr), 4),
        "grouped_pr_auc": round(float(grouped_pr), 4),
        "inflation": round(float(inflation), 4),
    },
    "traps_removed": {
        "removed": ["interaction_clay_wet", "nonmonotone_evi", "threshold_power",
                    "state_effect", "leakage_bait"],
        "note": ("Planted traps were removed and the generator retuned for "
                 "accuracy. The headline below is higher because the LABEL is "
                 "more learnable, not because the model improved."),
        "latent_strength": gen_latent_strength,
        "observable_log_spread": gen_log_spread,
    },
    "demo_case_outcomes": outcomes,
    "demo_case_checks": {
        "confusion_held": held, "confusion_total": int(len(conf)),
        "pairs_held": int(sum(p["held"] for p in pairs)), "pairs_total": len(pairs),
        "pairs": pairs,
        "note": ("Narrative cases are excluded from both counts on purpose: "
                 "their expected_role names a lesson, not a confusion cell."),
    },
    "attribution": {
        "mean_shares": {k: round(float(v), 4) for k, v in shares.mean().items()},
        "dominant_counts": {k: int(v) for k, v in dominant.value_counts().items()},
    },
    "second_opinion": {
        "note": ("No second opinion touches `risk`. Condition enters `decision`: the "
                 "one-sided condition score over the 30-day telemetry block, "
                 "which the supervised model cannot train on because its label "
                 "covers 36 months. It enters as a rank BLEND, not a gate: a "
                 "gate reorders nothing and scored -0.0125 F1 on held-out seeds "
                 "against the blend's +0.0191. The isolation forest is "
                 "descriptive — it scores |deviation| and need is monotone. Satellite "
                 "change enters priority only when its measured necessary condition passes."),
        "novelty": {
            "scorable_towers": int(scorable.sum()),
            "roc_auc": novelty_ranking["roc_auc"],
            "pr_auc": novelty_ranking["pr_auc"],
            "spearman_with_model": round(agreement, 4),
        },
        "telemetry_roc_auc": round(raw_telem_auc, 4),
        "detectors_in_escalation_band": detectors,
        "blend_sweep": blend_sweep,
        "matched_budget": {"budget": budget, "lower_cut": lower_cut,
                           "ensemble": ensemble, "delta_f1": delta_f1},
        "blend_weight": CONDITION_BLEND_WEIGHT,
        "change_weight": CHANGE_BLEND_WEIGHT,
        "change_blend_sweep": change_blend_sweep,
        "change_verdict": change_verdict,
        "towers_swapped_in": int(escalated_mask.sum()),
        "swapped_in_hits": int(y[escalated_mask].sum()),
        "novelty_by_quintile": ood,
        "flag_coverage": flag_coverage,
        "verdict": verdict,
    },
    "features": FEATURES,
    "factor_groups": FACTOR_GROUPS,
}

full.save_model(str(MODEL_PATH))
write_json(REPORT_PATH, report)
print(f"wrote {MODEL_PATH}")
print(f"wrote {REPORT_PATH}")

## What would change with real data

The pipeline is the deliverable; the label is the placeholder. Given a few
hundred real work orders — `site_id`, `date`, `what_was_done` —
`prepare_maintenance_records.py` becomes an **ingest** rather than a
generator, `maintenance_labels.csv` keeps its exact schema, and this notebook
does not change at all. That substitution being a one-file change is the point
of building it this way, and is worth saying out loud in the demo.

Two limits that real data would not fix, and which belong in any honest reading:

* **Tower positions are OpenStreetMap features, not an operator asset
  register.** Per-state counts describe mapping density, not deployment, so no
  per-state rate here should be rendered as coverage.
* **96% of masts carry no radio tag**, so `equipment` was dropped as a model
  factor: the trained booster split on `radio` zero times and its attribution
  share was exactly 0.000. The generator still simulates an equipment cause, so
  the model carries an unmodelled hazard component on purpose — deleting it
  from the label to flatter the model is exactly the move this notebook exists
  to avoid. The fix is an asset register, not a better estimator.